In [2]:
import cv2 as cv
import numpy as np
import os

# File paths
yolo_path = r"E:\YOLOv3\New folder"
image_path = os.path.join(yolo_path, "dog.jpg")  # Test image

weights_path = os.path.join(yolo_path, "yolov3.weights")
cfg_path = os.path.join(yolo_path, "yolov3.cfg")
names_path = os.path.join(yolo_path, "coco.names")

# Load class names
with open(names_path, "r") as f:
    classes = [line.strip() for line in f.readlines()]

# Load YOLO network
yolo_net = cv.dnn.readNet(weights_path, cfg_path)
yolo_net.setPreferableBackend(cv.dnn.DNN_BACKEND_OPENCV)
yolo_net.setPreferableTarget(cv.dnn.DNN_TARGET_CPU)

# Get output layer names
output_layers = yolo_net.getUnconnectedOutLayersNames()

# Read input image
image = cv.imread(image_path)
if image is None:
    print("⚠️ Failed to load image.")
    exit()

# Resize image for better detection
image = cv.resize(image, (800, 600))
height, width, channels = image.shape

# Create blob from image
blob = cv.dnn.blobFromImage(image, 1/255.0, (416, 416), swapRB=True, crop=False)
yolo_net.setInput(blob)

# Run forward pass
outputs = yolo_net.forward(output_layers)

# Check output layers
for i, out in enumerate(outputs):
    print(f"🔍 Output layer {i} shape: {out.shape}")
    if out.shape[0] == 0:
        print("⚠️ No detections in this layer.")

# Lists for detection results
boxes = []
confidences = []
class_ids = []

# Thresholds (low for testing)
confidence_threshold = 0.01
nms_threshold = 0.3

# Process detections
for out in outputs:
    for detection in out:
        scores = detection[5:]
        class_id = np.argmax(scores)
        confidence = scores[class_id]

        if confidence > confidence_threshold:
            center_x = int(detection[0] * width)
            center_y = int(detection[1] * height)
            w = int(detection[2] * width)
            h = int(detection[3] * height)

            x = int(center_x - w / 2)
            y = int(center_y - h / 2)

            boxes.append([x, y, w, h])
            confidences.append(float(confidence))
            class_ids.append(class_id)

# Apply Non-Maximum Suppression
indexes = cv.dnn.NMSBoxes(boxes, confidences, confidence_threshold, nms_threshold)

print("✅ Number of detected objects:", len(indexes))

# Generate random colors for each class
colors = np.random.uniform(0, 255, size=(len(classes), 3))

# Draw bounding boxes
if len(indexes) > 0:
    for i in indexes.flatten():
        x, y, w, h = boxes[i]
        label = str(classes[class_ids[i]])
        confidence = confidences[i]
        color = colors[class_ids[i]]

        print(f"📦 {label} with confidence {confidence:.2f}")
        cv.rectangle(image, (x, y), (x + w, y + h), color, 2)
        cv.putText(image, f"{label} {confidence:.2f}", (x, y - 10),
                   cv.FONT_HERSHEY_PLAIN, 2, color, 2)
else:
    print("⚠️ No objects detected.")

# Show result
cv.imshow("YOLOv3 Detection", image)
cv.waitKey(0)
cv.destroyAllWindows()

🔍 Output layer 0 shape: (507, 85)
🔍 Output layer 1 shape: (2028, 85)
🔍 Output layer 2 shape: (8112, 85)
✅ Number of detected objects: 3
📦 dog with confidence 1.00
📦 bicycle with confidence 0.99
📦 truck with confidence 0.95
